In [ ]:
from utils.available_datasets import available_datasets

dataset_choice = available_datasets["PERSEVERE"]

In [ ]:
train_split = dataset_choice.preferred_train_split
data_dir = dataset_choice.data_dir
splits_filepath = dataset_choice.splits_filepath
ndim = dataset_choice.ndim
input_channels = dataset_choice.input_channels

In [ ]:
from tqdm import tqdm
import shutil
import os

img_dir = os.path.join(data_dir, "img")
gt_dir = os.path.join(data_dir, "gt")

img_filenames_list = [f for f in os.listdir(img_dir) if f.endswith(('.nii.gz'))]
gt_filenames_list = [f for f in os.listdir(gt_dir) if f.endswith(('.nii.gz'))]

img_idx_list = []
for img_filename in tqdm(img_filenames_list):
    img_i = int(img_filename.split(".")[0])
    img_id = f"PERSEVERE_{img_i:03d}"
    img_idx_list.append(img_id)

    new_filename = img_id + ".nii.gz"
    src_filepath = os.path.join(img_dir, img_filename)
    dst_filepath = os.path.join(img_dir, new_filename)
    os.rename(src_filepath, dst_filepath)

img_idx_list.sort()
print(len(img_idx_list))

gt_idx_list = []
for gt_filename in tqdm(gt_filenames_list):
    gt_i = int(gt_filename.split(".")[0])
    gt_id = f"PERSEVERE_{gt_i:03d}"
    gt_idx_list.append(gt_id)

    new_filename = gt_id + ".nii.gz"
    src_filepath = os.path.join(gt_dir, gt_filename)
    dst_filepath = os.path.join(gt_dir, new_filename)
    os.rename(src_filepath, dst_filepath)

gt_idx_list.sort()
print(len(gt_idx_list))

In [ ]:
common_ids = list(set(img_idx_list).intersection(set(gt_idx_list)))
print(len(common_ids))
common_ids.sort()
print(common_ids)

no_gt_img_ids = list(set(img_idx_list).difference(set(gt_idx_list)))
print(len(no_gt_img_ids))
no_gt_img_ids.sort()
print(no_gt_img_ids)

In [ ]:
no_gt_img_dir = os.path.join(data_dir, "remaining_img")
os.makedirs(no_gt_img_dir, exist_ok=True)

for no_gt_img_id in tqdm(no_gt_img_ids):
    filename = no_gt_img_id + ".nii.gz"
    src_filepath = os.path.join(img_dir, filename)
    dst_filepath = os.path.join(no_gt_img_dir, filename)
    shutil.move(src_filepath, dst_filepath)

In [ ]:
from sklearn.model_selection import train_test_split
import json

train_idx, test_idx = train_test_split(common_ids, test_size=0.2, random_state=42)
train_idx.sort()
test_idx.sort()

splits = {}
splits['train'] = train_idx
splits['test'] = test_idx

# === Save splits ===
with open(splits_filepath, 'w') as f:
    json.dump(splits, f, indent=4)

In [ ]:
from image_segmentation.data import ImageDatamodule

datamodule = ImageDatamodule(data_dir=data_dir, 
                             split_file_path=splits_filepath,
                             train_split_name=train_split,
                             val_split_ratio=0.2,
                             train_transforms=None,
                             val_transforms=None,
                             test_transforms=None,
                             num_workers=0,
                             train_batch_size=1,
                             val_batch_size=1,
                             seed=42,
                             shuffle_train=False,
                             save_resolved_split=True,
                             mask_input=False)
datamodule.setup()

In [ ]:
stats = datamodule.dataset.get_dataset_stats(
    ndim=ndim,
    input_channels=input_channels,
    split_name=train_split,
    split_indices=datamodule.train_indices
)
print(stats)

In [ ]:
from image_segmentation.data.data_viz import plot_batch

dataloader = datamodule.train_dataloader()

for i, batch in enumerate(dataloader):
    plot_batch(batch, ndim)
    if i >= 3:
        break